# Notebook 03: Mask Fine-Tuning with Classification Heads

In this notebook, we perform end-to-end training but **freeze the entire Moirai encoder except for the mask tokens**. 

In [1]:
import torch
import pandas as pd
from IPython.display import display

from heads import (
    MeanPoolingClassifier,
    SingleScaleAttentionClassifier,
    SingleScaleMultiHeadClassifier,
    HierarchicalMultiHeadClassifier,
)
from models import MaskOnlyFinetunerWrapper, HeadFinetunerWrapper
from utils import get_lsst_dataloaders, universal_grid_search

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [8, 16, 32, 64]

lr_grid = [1e-4]
wd_grid = [0.01, 0.05]

BATCH_SIZE = 256

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)

In [3]:
METRICS_COLS = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted Precision", "Weighted Recall", "Weighted F1"]
df_patch_8_metrics = pd.DataFrame(columns=METRICS_COLS)
df_patch_8_metrics.index.name = "Model"

## 1. Baseline: Linear Model on Mask Embedding Only

In [4]:
df_mask_only = pd.DataFrame(index=["Mask Only"], columns=PATCH_SIZES)
df_mask_only.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    print(f"Patch {patch}")
    _, metrics = universal_grid_search(
        model_class=MaskOnlyFinetunerWrapper,
        model_kwargs={"patch_size": patch, "num_vars": NUM_VARS, "num_classes": num_classes, "size": SIZE},
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=wd_grid, lr_grid=lr_grid
    )
    df_mask_only.loc["Mask Only", patch] = metrics["Accuracy"]
    if patch == 8:
        df_patch_8_metrics.loc["Mask Only"] = metrics

Patch 8
LR=0.0001| WD=0.01
 Early stopping : 231
Val Loss: 1.3524
LR=0.0001| WD=0.05
 Early stopping : 218
Val Loss: 1.3518
Patch 16
LR=0.0001| WD=0.01
 Early stopping : 213
Val Loss: 1.2648
LR=0.0001| WD=0.05
 Early stopping : 210
Val Loss: 1.2668
Patch 32
LR=0.0001| WD=0.01


 Early stopping : 201
Val Loss: 1.3249
LR=0.0001| WD=0.05
 Early stopping : 197
Val Loss: 1.3209
Patch 64
LR=0.0001| WD=0.01
 Early stopping : 186
Val Loss: 1.2412
LR=0.0001| WD=0.05
 Early stopping : 190
Val Loss: 1.2332


In [5]:
print("Mask Only - Accuracy")
display(df_mask_only.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Mask Only - Accuracy


Patch Size,8,16,32,64
Mask Only,0.5612,0.5831,0.5892,0.5884



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.5612,0.3684,0.3078,0.3115,0.5026,0.5612,0.5108


## 2. Baseline: Mean Pooling on Full Sequence (Context + Mask)
We average all patches (including the unfrozen mask) before passing them to a linear classifier.

In [6]:
df_mean_pool = pd.DataFrame(index=["Mean Pooling"], columns=PATCH_SIZES)
df_mean_pool.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, metrics = universal_grid_search(
        model_class=HeadFinetunerWrapper,
        model_kwargs={
            "head_class": MeanPoolingClassifier,
            "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
            "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=wd_grid, lr_grid=lr_grid
    )
    df_mean_pool.loc["Mean Pooling", patch] = metrics["Accuracy"]
    if patch == 8:
        df_patch_8_metrics.loc["Mean Pooling"] = metrics

LR=0.0001| WD=0.01


KeyboardInterrupt: 

In [ ]:
print("Mean Pooling - Accuracy")
display(df_mean_pool.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Mean Pooling - Accuracy


Patch Size,8,16,32,64
Mean Pooling,0.6054,0.6221,0.6091,0.6225



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.5645,0.3664,0.3140,0.3169,0.5058,0.5645,0.5179
Mean Pooling,0.6054,0.4300,0.3644,0.3733,0.5525,0.6054,0.5632


## 3. Advanced Pooling: Attention & MHA (Context + Mask)
We use Attention mechanisms to dynamically weight the patches. The network can now learn to pay attention to specific context patches AND the fine-tuned mask patch.

In [ ]:
PATCH_SIZES_TO_TEST = [8, 16, 32, 64]
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 16

model_names_single = ["Attention (Base)", f"MHA (Heads={NUM_HEADS})"]
index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Model", "Mode"])
df_adv_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_adv_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        # 1. Basic Attention
        _, metrics_attn = universal_grid_search(
            model_class=HeadFinetunerWrapper,
            model_kwargs={
                "head_class": SingleScaleAttentionClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[("Attention (Base)", mode), patch] = metrics_attn["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"Attention ({mode})"] = metrics_attn

        # 2. Multi-Head Attention
        _, metrics_mha = universal_grid_search(
            model_class=HeadFinetunerWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[(f"MHA (Heads={NUM_HEADS})", mode), patch] = metrics_mha["Accuracy"]

LR=0.001| WD=0.05
 Early stopping : 44
Val Loss: 1.1758
LR=0.001| WD=0.05
 Early stopping : 18
Val Loss: 1.1416
LR=0.001| WD=0.05
 Early stopping : 35
Val Loss: 1.1809
LR=0.001| WD=0.05
 Early stopping : 16
Val Loss: 1.1558
LR=0.001| WD=0.05
 Early stopping : 31
Val Loss: 1.1686
LR=0.001| WD=0.05
 Early stopping : 11
Val Loss: 1.1273
LR=0.001| WD=0.05
 Early stopping : 27
Val Loss: 1.1806
LR=0.001| WD=0.05
 Early stopping : 14
Val Loss: 1.1327
LR=0.001| WD=0.05
 Early stopping : 27
Val Loss: 1.2582
LR=0.001| WD=0.05
 Early stopping : 13
Val Loss: 1.1894
LR=0.001| WD=0.05
 Early stopping : 25
Val Loss: 1.2374
LR=0.001| WD=0.05
 Early stopping : 15
Val Loss: 1.1918
LR=0.001| WD=0.05
 Early stopping : 36
Val Loss: 1.1495
LR=0.001| WD=0.05
 Early stopping : 14
Val Loss: 1.1257
LR=0.001| WD=0.05
 Early stopping : 34
Val Loss: 1.1338
LR=0.001| WD=0.05
 Early stopping : 12
Val Loss: 1.1321


In [ ]:
print("Attention & MHA - Accuracy")
display(df_adv_single.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Attention & MHA - Accuracy


Patch Size                                8       16      32      64
Model            Mode                                               
Attention (Base) shared_context       0.6087  0.6002  0.5823  0.6261
                 independent_context  0.5998  0.6095  0.5921  0.6212
MHA (Heads=16)   shared_context       0.6115  0.6071  0.5787  0.6225
                 independent_context  0.6071  0.6030  0.5989  0.6095


Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.5645,0.3664,0.3140,0.3169,0.5058,0.5645,0.5179
Mean Pooling,0.6054,0.4300,0.3644,0.3733,0.5525,0.6054,0.5632
Attention (shared_context),0.6087,0.4909,0.3636,0.3728,0.5633,0.6087,0.5619
Attention (independent_context),0.5998,0.4959,0.3607,0.3698,0.5552,0.5998,0.5544


## 4. Ablation Study: Number of Attention Heads
Testing the impact of `num_heads` on the Single-Scale MHA model (with mask fine-tuning) for a fixed patch size of 16.

In [ ]:
MODES = ["shared_context", "independent_context"]
HEADS_TO_TEST = [4, 8, 16, 32, 64, 128, 384]

df_heads_ablation8 = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_ablation8.index.name = "Mode"
df_heads_ablation8.columns.name = "Num Heads (Patch 8)"

for PATCH in [8]:
    for mode in MODES:
        for heads in HEADS_TO_TEST:
            print(f'{mode} - {heads}')
            _, metrics = universal_grid_search(
                model_class=HeadFinetunerWrapper,
                model_kwargs={
                    "head_class": SingleScaleMultiHeadClassifier,
                    "head_kwargs": {
                        "num_vars": NUM_VARS,
                        "num_classes": num_classes,
                        "patch_mode": mode,
                        "num_heads": heads
                    },
                    "patch_size": PATCH,
                    "num_vars": NUM_VARS,
                    "size": SIZE,
                    "remove_last_patch": False
                },
                train_loader=tr_loader,
                val_loader=va_loader,
                test_loader=te_loader,
                device=DEVICE,
                wd_grid=wd_grid, lr_grid=lr_grid
            )
            df_heads_ablation8.loc[mode, heads] = metrics["Accuracy"]
            df_patch_8_metrics.loc[f"MHA-{heads} ({mode})"] = metrics

LR=0.001| WD=0.05
 Early stopping : 16
Val Loss: 1.1507
LR=0.001| WD=0.05
 Early stopping : 14
Val Loss: 1.1334
LR=0.001| WD=0.05
 Early stopping : 12
Val Loss: 1.1601
LR=0.001| WD=0.05
 Early stopping : 14
Val Loss: 1.1462
LR=0.001| WD=0.05
 Early stopping : 12
Val Loss: 1.1629
LR=0.001| WD=0.05
 Early stopping : 15
Val Loss: 1.1423
LR=0.001| WD=0.05
 Early stopping : 18
Val Loss: 1.1578
LR=0.001| WD=0.05
 Early stopping : 15
Val Loss: 1.1583
LR=0.001| WD=0.05
 Early stopping : 14
Val Loss: 1.1080
LR=0.001| WD=0.05
 Early stopping : 13
Val Loss: 1.1114
LR=0.001| WD=0.05
 Early stopping : 13
Val Loss: 1.1044
LR=0.001| WD=0.05
 Early stopping : 13
Val Loss: 1.1301
LR=0.001| WD=0.05
 Early stopping : 16
Val Loss: 1.1324
LR=0.001| WD=0.05
 Early stopping : 13
Val Loss: 1.1226
LR=0.001| WD=0.05
 Early stopping : 14
Val Loss: 1.1328
LR=0.001| WD=0.05
 Early stopping : 16
Val Loss: 1.1352


In [ ]:
print("Ablation: Num Heads (Patch = 8) - Accuracy")
display(df_heads_ablation8.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Ablation: Num Heads (Patch = 8) - Accuracy


Num Heads (Patch 8),16,32,64,128
Mode,,,,
shared_context,0.6091,0.6075,0.6050,0.6010
independent_context,0.5973,0.6018,0.6038,0.6018


Ablation: Num Heads (Patch = 16) - Accuracy


Num Heads (Patch 16),16,32,64,128
Mode,,,,
shared_context,0.618,0.6071,0.6123,0.6249
independent_context,0.603,0.6196,0.6119,0.6139



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.5645,0.3664,0.3140,0.3169,0.5058,0.5645,0.5179
Mean Pooling,0.6054,0.4300,0.3644,0.3733,0.5525,0.6054,0.5632
Attention (shared_context),0.6087,0.4909,0.3636,0.3728,0.5633,0.6087,0.5619
Attention (independent_context),0.5998,0.4959,0.3607,0.3698,0.5552,0.5998,0.5544
MHA-16 (shared_context),0.6091,0.4482,0.4043,0.4086,0.5615,0.6091,0.5733
MHA-32 (shared_context),0.6075,0.4057,0.3634,0.3695,0.5501,0.6075,0.5615
MHA-64 (shared_context),0.6050,0.4333,0.3629,0.3751,0.5589,0.6050,0.5617
MHA-128 (shared_context),0.6010,0.4709,0.3752,0.3869,0.5507,0.6010,0.5640
MHA-16 (independent_context),0.5973,0.4073,0.3574,0.3610,0.5389,0.5973,0.5518


# 5. Hierarchical

In [ ]:
df_hierarchical = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_hierarchical.index.name = "Mode"
df_hierarchical.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, metrics_hier = universal_grid_search(
            model_class=HeadFinetunerWrapper,
            model_kwargs={
                "head_class": HierarchicalMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_hierarchical.loc[mode, patch] = metrics_hier["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"Hierarchical ({mode})"] = metrics_hier

LR=0.001| WD=0.05
 Early stopping : 22
Val Loss: 1.2489
LR=0.001| WD=0.05
 Early stopping : 21
Val Loss: 1.1960
LR=0.001| WD=0.05
 Early stopping : 17
Val Loss: 1.2673
LR=0.001| WD=0.05
 Early stopping : 15
Val Loss: 1.2309
LR=0.001| WD=0.05
 Early stopping : 18
Val Loss: 1.2714
LR=0.001| WD=0.05
 Early stopping : 15
Val Loss: 1.3123
LR=0.001| WD=0.05
 Early stopping : 16
Val Loss: 1.2572
LR=0.001| WD=0.05
 Early stopping : 15
Val Loss: 1.2716


In [ ]:
print("Hierarchical MHA - Accuracy")
display(df_hierarchical.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Hierarchical MHA - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.5799,0.5949,0.5779,0.5762
independent_context,0.6038,0.5839,0.5661,0.5730



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.5645,0.3664,0.3140,0.3169,0.5058,0.5645,0.5179
Mean Pooling,0.6054,0.4300,0.3644,0.3733,0.5525,0.6054,0.5632
Attention (shared_context),0.6087,0.4909,0.3636,0.3728,0.5633,0.6087,0.5619
Attention (independent_context),0.5998,0.4959,0.3607,0.3698,0.5552,0.5998,0.5544
MHA-16 (shared_context),0.6091,0.4482,0.4043,0.4086,0.5615,0.6091,0.5733
MHA-32 (shared_context),0.6075,0.4057,0.3634,0.3695,0.5501,0.6075,0.5615
MHA-64 (shared_context),0.6050,0.4333,0.3629,0.3751,0.5589,0.6050,0.5617
MHA-128 (shared_context),0.6010,0.4709,0.3752,0.3869,0.5507,0.6010,0.5640
MHA-16 (independent_context),0.5973,0.4073,0.3574,0.3610,0.5389,0.5973,0.5518
